# UTS Machine Learning - Klasifikasi
**NIM 24552011338** → jumlah digit 34 → 34 mod 10 = 4 → **Kasus 4: Bayi Berat Lahir Rendah** (`kasus04_birthwt.csv`, target `low`).

**Alur:** (1) siapkan data & audit → (2) `fitur.py` → (3) `latih.py` (split → CV → threshold → test 1x → joblib) → (4) `app.py` Streamlit → (5) jalankan di Colab → (6) laporan.

Jalankan sel dari atas ke bawah.

In [3]:
!pip -q install streamlit joblib scikit-learn pandas statsmodels tabulate
import os
os.makedirs('dataset', exist_ok=True); os.makedirs('models', exist_ok=True); os.makedirs('notebook', exist_ok=True)

## 1. Muat dataset
Unggah `kasus04_birthwt.csv` saat diminta. Cadangan: GitHub dosen, lalu `MASS::birthwt`.

In [4]:
import pandas as pd, urllib.request, os
PATH = 'dataset/kasus04_birthwt.csv'

# Cara paling aman: unggah file kasus04_birthwt.csv milik Anda (klik "Choose Files").
# Jika tidak memilih file, notebook mencoba GitHub dosen lalu dataset MASS::birthwt.
try:
    from google.colab import files
    up = files.upload()
    if up:
        open(PATH, 'wb').write(list(up.values())[0])
except Exception as e:
    print('Upload dilewati:', type(e).__name__)

if not os.path.exists(PATH):
    for b in ['main', 'master']:
        u = f'https://raw.githubusercontent.com/Muhammad-Ikhwan-Fathulloh/ArtificialIntelegent-Course-Bank/{b}/UTS/dataset/kasus04_birthwt.csv'
        try:
            raw = urllib.request.urlopen(u, timeout=20).read().decode()
            if 'low' in raw.splitlines()[0]:
                open(PATH, 'w').write(raw); print('Dari GitHub:', u); break
        except Exception as e:
            print('Gagal:', u, type(e).__name__)
if not os.path.exists(PATH):
    import statsmodels.api as sm
    sm.datasets.get_rdataset('birthwt', 'MASS').data.to_csv(PATH, index=False)
    print('Dari Rdatasets (MASS::birthwt)')
df = pd.read_csv(PATH); print(df.shape); df.head()

Saving kasus04_birthwt.csv to kasus04_birthwt.csv
(189, 10)


,low,age,lwt,race,smoke,ptl,ht,ui,ftv,bwt
0,0,19,182,2,0,0,0,1,0,2523
1,0,33,155,3,0,0,0,0,3,2551
2,0,20,105,1,1,0,0,0,1,2557
3,0,21,108,1,1,0,0,1,2,2594
4,0,18,107,1,1,0,0,1,0,2600


## 2. Audit data (tahap 1)

In [5]:
print('Duplikat:', df.duplicated().sum())
print('\nMissing:\n', df.isna().sum())
print('\nTipe:\n', df.dtypes)
display(df.describe().T)
print('\nDistribusi target low:\n', df['low'].value_counts(normalize=True).round(3))
# Bukti kebocoran: low == (bwt < 2500) -> bwt WAJIB dibuang
print('\nlow identik dengan (bwt<2500)?', ((df['bwt'] < 2500).astype(int) == df['low']).all())
display(df.groupby('race')['low'].agg(['mean', 'count']).rename(columns={'mean': 'proporsi_low'}))

Duplikat: 5

Missing:
 low      0
age      0
lwt      0
race     0
smoke    0
ptl      0
ht       0
ui       0
ftv      0
bwt      0
dtype: int64

Tipe:
 low      int64
age      int64
lwt      int64
race     int64
smoke    int64
ptl      int64
ht       int64
ui       int64
ftv      int64
bwt      int64
dtype: object


,count,mean,std,min,25%,50%,75%,max
low,189.0,0.312169,0.464609,0.0,0.0,0.0,1.0,1.0
age,189.0,23.238095,5.298678,14.0,19.0,23.0,26.0,45.0
lwt,189.0,129.814815,30.579380,80.0,110.0,121.0,140.0,250.0
race,189.0,1.846561,0.918342,1.0,1.0,1.0,3.0,3.0
smoke,189.0,0.391534,0.489390,0.0,0.0,0.0,1.0,1.0
ptl,189.0,0.195767,0.493342,0.0,0.0,0.0,0.0,3.0
ht,189.0,0.063492,0.244494,0.0,0.0,0.0,0.0,1.0
ui,189.0,0.148148,0.356190,0.0,0.0,0.0,0.0,1.0
ftv,189.0,0.793651,1.059286,0.0,0.0,0.0,1.0,6.0
bwt,189.0,2944.587302,729.214295,709.0,2414.0,2977.0,3487.0,4990.0



Distribusi target low:
 low
0    0.688
1    0.312
Name: proportion, dtype: float64

low identik dengan (bwt<2500)? True


,proporsi_low,count
race,,
1,0.239583,96
2,0.423077,26
3,0.373134,67


### Keputusan audit
| Kolom | Keputusan | Alasan |
|---|---|---|
| `bwt` | **Dibuang** | Berat lahir baru diketahui setelah lahir, dan `low = bwt < 2500` → target bocor (leakage). |
| `rownames`/ID | **Dibuang** | Hanya nomor baris. |
| `race` | **Kategorik** (one-hot) | Kode 1/2/3 nominal, tanpa urutan. Jika dibiarkan numerik, model mengira ras 3 "lebih besar" dari 1. |
| `age, lwt, ptl, ftv` | Numerik, imputasi median + scaling | Skala berbeda; statistik dihitung dari train saja. |
| `smoke, ht, ui` | Biner 0/1 | Sudah sesuai. |
| Fitur baru | `ptl_pernah`, `ftv_pernah`, `usia_berisiko` (<18 / ≥35), `lwt_rendah` (<100 pon), `jumlah_risiko` | Aturan domain tetap (tanpa statistik data) sehingga tidak ada leakage; menangkap efek non-linear & akumulasi risiko. |
| Duplikat | **5 baris duplikat persis dibuang** (189 → 184) | Pasangan baris berurutan yang identik di semua kolom termasuk `bwt`, indikasi input ganda. Jika dibiarkan, salinannya bisa jatuh di train dan test sehingga skor test terlalu optimis. |
| Missing / tidak wajar | Tidak ada missing dan tidak ada nilai di luar rentang; imputer tetap dipasang agar aplikasi robust | Rentang divalidasi di `audit_data`. |

## 3. `fitur.py`

In [6]:
%%writefile fitur.py
"""fitur.py - Feature engineering Kasus 4 (birthwt, target = low)."""
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

TARGET = "low"
KOLOM_ASLI = ["age", "lwt", "race", "smoke", "ptl", "ht", "ui", "ftv"]

# Kolom yang DIBUANG beserta alasannya
KOLOM_DIBUANG = {
    "bwt": "Berat lahir bayi (gram). Baru diketahui setelah bayi lahir dan low = (bwt < 2500). "
           "Memakainya adalah data leakage (target bocor).",
    "rownames/Unnamed: 0/id": "Hanya nomor baris/ID, tidak punya makna prediktif.",
}

RACE_LABEL = {1: "putih", 2: "hitam", 3: "lainnya"}

NUMERIK = ["age", "lwt", "ptl", "ftv"]
BINER = ["smoke", "ht", "ui", "ptl_pernah", "ftv_pernah",
         "usia_berisiko", "lwt_rendah", "jumlah_risiko"]
KATEGORIK = ["race_label"]


def muat_data(path):
    """Baca CSV, buang kolom ID/indeks, dan buang duplikat persis."""
    df = pd.read_csv(path)
    for c in list(df.columns):
        if c.lower() in ("rownames", "unnamed: 0", "id", "index", ""):
            df = df.drop(columns=c)
    return df.drop_duplicates().reset_index(drop=True)


def audit_data(df):
    """Ringkasan audit: duplikat, missing, tipe data, nilai tidak wajar."""
    return {
        "bentuk": df.shape,
        "duplikat": int(df.duplicated().sum()),
        "missing": df.isna().sum().to_dict(),
        "tipe": df.dtypes.astype(str).to_dict(),
        "tidak_wajar": {
            "age di luar 12-50": int((~df["age"].between(12, 50)).sum()),
            "lwt di luar 70-400 pon": int((~df["lwt"].between(70, 400)).sum()),
            "biner bukan 0/1": int(sum((~df[c].isin([0, 1])).sum() for c in ["smoke", "ht", "ui"])),
            "race di luar 1-3": int((~df["race"].isin([1, 2, 3])).sum()),
        },
    }


def rekayasa_fitur(X):
    """Fitur turunan TANPA statistik data (aturan domain tetap) -> aman dari leakage."""
    X = X.copy()
    # race berupa kode nominal (bukan urutan) -> jadikan kategorik
    X["race_label"] = X["race"].map(RACE_LABEL).fillna("lainnya")
    X["ptl_pernah"] = (X["ptl"] > 0).astype(int)           # pernah lahir prematur
    X["ftv_pernah"] = (X["ftv"] > 0).astype(int)           # pernah periksa trimester 1
    X["usia_berisiko"] = ((X["age"] < 18) | (X["age"] >= 35)).astype(int)  # remaja / usia lanjut
    X["lwt_rendah"] = (X["lwt"] < 100).astype(int)         # BB ibu < 100 pon (~45 kg)
    X["jumlah_risiko"] = X["smoke"] + X["ht"] + X["ui"] + X["ptl_pernah"]  # akumulasi risiko
    return X[NUMERIK + BINER + KATEGORIK]


def buat_preprocessor():
    num = Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())])
    bin_ = Pipeline([("imp", SimpleImputer(strategy="most_frequent"))])
    kat = Pipeline([("oh", OneHotEncoder(handle_unknown="ignore"))])
    return ColumnTransformer([("num", num, NUMERIK), ("bin", bin_, BINER), ("kat", kat, KATEGORIK)])


def buat_pipeline(model):
    return Pipeline([
        ("fitur", FunctionTransformer(rekayasa_fitur, validate=False)),
        ("prep", buat_preprocessor()),
        ("model", model),
    ])

Writing fitur.py


## 4. `latih.py`
Split stratified 80/20 **sebelum** preprocessing → CV (Repeated Stratified 5x3) di train untuk memilih model → threshold dari prediksi out-of-fold → test dievaluasi **satu kali** → simpan joblib.

In [7]:
%%writefile latih.py
"""latih.py - Training, evaluasi, dan penyimpanan model Kasus 4."""
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (make_scorer, average_precision_score, classification_report, confusion_matrix,
                             f1_score, fbeta_score, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import (RepeatedStratifiedKFold, StratifiedKFold, cross_val_predict,
                                     cross_validate, train_test_split)

import fitur as F

SEED = 42
TARGET_RECALL = 0.80
BASE = Path(__file__).parent if "__file__" in globals() else Path(".")
DATA = BASE / "dataset" / "kasus04_birthwt.csv"
MODEL_DIR = BASE / "models"
MODEL_DIR.mkdir(exist_ok=True)


def main():
    df = F.muat_data(DATA)
    print("Audit:", F.audit_data(df))

    # 1) SPLIT DULU (sebelum imputasi/scaling/encoding)
    X = df[F.KOLOM_ASLI]
    y = df[F.TARGET].astype(int)          # 'bwt' tidak masuk X (leakage)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
    print(f"Train {X_tr.shape}, Test {X_te.shape}, proporsi low train={y_tr.mean():.2f} test={y_te.mean():.2f}")

    # 2) Kandidat model
    kandidat = {
        "Dummy (baseline)": DummyClassifier(strategy="prior"),
        "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced", C=0.5),
        "Random Forest": RandomForestClassifier(n_estimators=400, min_samples_leaf=5, max_depth=5,
                                                class_weight="balanced_subsample",
                                                random_state=SEED, n_jobs=-1),
    }

    # 3) Pemilihan model: cross-validation di data TRAIN saja
    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)
    skor = {"pr_auc": "average_precision", "roc_auc": "roc_auc", "recall": "recall",
            "precision": make_scorer(precision_score, zero_division=0),
            "f1": make_scorer(f1_score, zero_division=0)}
    baris = []
    for nama, m in kandidat.items():
        r = cross_validate(F.buat_pipeline(m), X_tr, y_tr, cv=cv, scoring=skor, n_jobs=-1)
        baris.append({"model": nama, **{k: float(np.mean(r[f"test_{k}"])) for k in skor},
                      **{f"{k}_std": float(np.std(r[f"test_{k}"])) for k in ("pr_auc",)}})
    tabel_cv = pd.DataFrame(baris).set_index("model")
    print("\nHasil Cross-Validation (train):\n", tabel_cv.round(3))

    layak = tabel_cv.drop(index="Dummy (baseline)")
    nama_terbaik = layak["pr_auc"].idxmax()          # ranking bebas threshold
    print("\nModel terpilih:", nama_terbaik)

    # 4) Threshold dipilih dari prediksi out-of-fold di TRAIN (bukan dari test).
    #    Aturan: recall minimal 0.80 (FN lebih mahal), lalu ambil threshold TERTINGGI
    #    yang masih memenuhi supaya false positive sesedikit mungkin.
    pipa = F.buat_pipeline(kandidat[nama_terbaik])
    oof = cross_val_predict(pipa, X_tr, y_tr, cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
                            method="predict_proba")[:, 1]
    grid = np.round(np.arange(0.05, 0.96, 0.01), 2)
    rec = np.array([recall_score(y_tr, oof >= t) for t in grid])
    ok = np.where(rec >= TARGET_RECALL)[0]
    if len(ok):
        thr = float(grid[ok[-1]])
    else:  # cadangan: maksimalkan F2
        thr = float(grid[int(np.argmax([fbeta_score(y_tr, oof >= t, beta=2) for t in grid]))])
    print(f"Threshold terpilih = {thr} (recall OOF = {recall_score(y_tr, oof >= thr):.2f})")

    # 5) Evaluasi data test SATU kali
    pipa.fit(X_tr, y_tr)
    proba = pipa.predict_proba(X_te)[:, 1]
    pred = (proba >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_te, pred, labels=[0, 1]).ravel()
    metrik = {
        "recall": recall_score(y_te, pred), "precision": precision_score(y_te, pred, zero_division=0),
        "f1": f1_score(y_te, pred), "f2": fbeta_score(y_te, pred, beta=2),
        "roc_auc": roc_auc_score(y_te, proba), "pr_auc": average_precision_score(y_te, proba),
        "specificity": tn / (tn + fp), "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
    }
    metrik = {k: float(v) for k, v in metrik.items()}
    print("\nMetrik TEST:", {k: round(v, 3) for k, v in metrik.items()})
    print(classification_report(y_te, pred, target_names=["Normal (0)", "BBLR (1)"], zero_division=0))

    # 6) Interpretasi fitur
    nama_fitur = pipa.named_steps["prep"].get_feature_names_out()
    est = pipa.named_steps["model"]
    imp = est.coef_[0] if hasattr(est, "coef_") else est.feature_importances_
    penting = dict(sorted(zip(map(str, nama_fitur), map(float, imp)), key=lambda kv: -abs(kv[1])))

    # 7) Simpan pipeline lengkap + metadata
    paket = {
        "pipeline": pipa, "threshold": thr, "target_recall": TARGET_RECALL, "model_name": nama_terbaik,
        "metrics_test": metrik, "cv_table": tabel_cv.round(4).to_dict("index"),
        "feature_importance": penting, "input_columns": F.KOLOM_ASLI,
        "race_label": F.RACE_LABEL, "kolom_dibuang": F.KOLOM_DIBUANG,
        "target_meaning": {0: "Berat lahir normal (>= 2500 g)", 1: "Berat lahir rendah (< 2500 g)"},
        "sklearn_version": sklearn.__version__, "seed": SEED,
        "n_train": int(len(X_tr)), "n_test": int(len(X_te)),
    }
    joblib.dump(paket, MODEL_DIR / "model_bblr.joblib")
    (MODEL_DIR / "metrik.json").write_text(json.dumps(
        {"model": nama_terbaik, "threshold": thr, "test": metrik, "cv": paket["cv_table"],
         "fitur_penting": penting}, indent=2))
    print("\nTersimpan di models/model_bblr.joblib")


if __name__ == "__main__":
    main()

Writing latih.py


In [8]:
!python latih.py

Audit: {'bentuk': (184, 10), 'duplikat': 0, 'missing': {'low': 0, 'age': 0, 'lwt': 0, 'race': 0, 'smoke': 0, 'ptl': 0, 'ht': 0, 'ui': 0, 'ftv': 0, 'bwt': 0}, 'tipe': {'low': 'int64', 'age': 'int64', 'lwt': 'int64', 'race': 'int64', 'smoke': 'int64', 'ptl': 'int64', 'ht': 'int64', 'ui': 'int64', 'ftv': 'int64', 'bwt': 'int64'}, 'tidak_wajar': {'age di luar 12-50': 0, 'lwt di luar 70-400 pon': 0, 'biner bukan 0/1': 0, 'race di luar 1-3': 0}}
Train (147, 8), Test (37, 8), proporsi low train=0.32 test=0.32

Hasil Cross-Validation (train):
                      pr_auc  roc_auc  recall  precision     f1  pr_auc_std
model                                                                     
Dummy (baseline)      0.320    0.500   0.000      0.000  0.000       0.011
Logistic Regression   0.501    0.644   0.520      0.400  0.449       0.132
Random Forest         0.512    0.677   0.481      0.405  0.434       0.091

Model terpilih: Random Forest
Threshold terpilih = 0.38 (recall OOF = 0.83)

Metri

## 5. Metrik evaluasi & interpretasi
**Kesalahan paling mahal = False Negative** (bayi berisiko BBLR diprediksi normal → ibu tidak mendapat pemantauan ekstra). False Positive hanya berarti pemeriksaan tambahan.

- **Metrik utama: Recall (sensitivitas)**, dengan target ≥ 0,80, lalu precision dijaga setinggi mungkin.
- **Threshold** diturunkan dari 0,5 ke nilai yang dipilih dari prediksi out-of-fold *di data train* (bukan dari test).
- Pendukung: **PR-AUC** (dipakai memilih model, karena kelas tidak seimbang ±31% positif), ROC-AUC, F1/F2, specificity, confusion matrix.
- **Akurasi tidak dipakai sebagai metrik utama** karena menebak "normal" semua sudah ±69% benar.

In [9]:
import json, joblib, pandas as pd
import fitur  # agar pipeline bisa di-load
paket = joblib.load('models/model_bblr.joblib')
print('Model:', paket['model_name'], '| threshold:', paket['threshold'])
display(pd.DataFrame(paket['cv_table']).T)
display(pd.Series(paket['metrics_test']).to_frame('TEST').T)
cm = paket['metrics_test']
display(pd.DataFrame([[cm['TN'], cm['FP']], [cm['FN'], cm['TP']]], index=['Aktual Normal', 'Aktual BBLR'], columns=['Pred Normal', 'Pred BBLR']).astype(int))
display(pd.Series(paket['feature_importance']).head(10).to_frame('koef/importance'))

Model: Random Forest | threshold: 0.38


,pr_auc,roc_auc,recall,precision,f1,pr_auc_std
Dummy (baseline),0.3195,0.5000,0.0000,0.0000,0.0000,0.0113
Logistic Regression,0.5006,0.6437,0.5200,0.4003,0.4485,0.1317
Random Forest,0.5120,0.6773,0.4815,0.4048,0.4344,0.0913


,recall,precision,f1,f2,roc_auc,pr_auc,specificity,TN,FP,FN,TP
TEST,0.916667,0.407407,0.564103,0.733333,0.693333,0.507161,0.36,9.0,16.0,1.0,11.0


,Pred Normal,Pred BBLR
Aktual Normal,9,16
Aktual BBLR,1,11


,koef/importance
num__lwt,0.205384
num__age,0.182176
bin__jumlah_risiko,0.142081
kat__race_label_putih,0.090045
num__ptl,0.063545
bin__smoke,0.052736
num__ftv,0.047605
bin__ptl_pernah,0.046972
kat__race_label_lainnya,0.035876
bin__ftv_pernah,0.034880


## 6. `app.py` (Streamlit)

In [10]:
%%writefile app.py
"""app.py - Aplikasi Streamlit prediksi Berat Lahir Rendah (Kasus 4)."""
from pathlib import Path

import joblib
import pandas as pd
import streamlit as st

import fitur  # noqa: F401  (diperlukan agar pipeline bisa di-load)

MODEL_PATH = Path(__file__).parent / "models" / "model_bblr.joblib"
st.set_page_config(page_title="Prediksi Berat Lahir Rendah", page_icon="👶")


@st.cache_resource
def muat_model():
    return joblib.load(MODEL_PATH)


st.title("👶 Prediksi Bayi Berat Lahir Rendah (BBLR)")
st.caption("Alat bantu skrining berbasis data 189 kelahiran. Bukan pengganti diagnosis dokter.")

if not MODEL_PATH.exists():
    st.error("Model belum ada. Jalankan `python latih.py` terlebih dahulu.")
    st.stop()

paket = muat_model()
tab1, tab2 = st.tabs(["🔮 Prediksi", "📊 Evaluasi Model"])

with tab1:
    c1, c2 = st.columns(2)
    with c1:
        age = st.number_input("Usia ibu (tahun)", 12, 50, 25)
        lwt = st.number_input("Berat badan ibu saat haid terakhir (pon)", 70, 400, 130)
        race_nama = st.selectbox("Ras ibu", list(paket["race_label"].values()))
        ptl = st.number_input("Jumlah kelahiran prematur sebelumnya", 0, 10, 0)
    with c2:
        smoke = st.selectbox("Merokok saat hamil?", ["Tidak", "Ya"])
        ht = st.selectbox("Riwayat hipertensi?", ["Tidak", "Ya"])
        ui = st.selectbox("Iritabilitas uterus?", ["Tidak", "Ya"])
        ftv = st.number_input("Kunjungan dokter trimester 1", 0, 10, 0)

    if st.button("Prediksi", type="primary"):
        kode_race = {v: k for k, v in paket["race_label"].items()}[race_nama]
        raw = pd.DataFrame([{
            "age": age, "lwt": lwt, "race": kode_race, "smoke": int(smoke == "Ya"),
            "ptl": ptl, "ht": int(ht == "Ya"), "ui": int(ui == "Ya"), "ftv": ftv,
        }])[paket["input_columns"]]
        p = float(paket["pipeline"].predict_proba(raw)[0, 1])
        thr = paket["threshold"]
        st.metric("Probabilitas BBLR", f"{p:.1%}", help=f"Threshold keputusan = {thr:.2f}")
        st.progress(min(max(p, 0.0), 1.0))
        if p >= thr:
            st.warning("⚠️ **Berisiko BBLR** - disarankan pemantauan/konsultasi lebih lanjut.")
        else:
            st.success("✅ **Risiko rendah** untuk BBLR.")
        st.write(f"Probabilitas Normal: **{1 - p:.1%}**")
        with st.expander("Data mentah yang dikirim ke pipeline"):
            st.dataframe(raw)

with tab2:
    st.subheader(f"Model terpilih: {paket['model_name']}")
    m = paket["metrics_test"]
    a, b, c, d = st.columns(4)
    a.metric("Recall", f"{m['recall']:.2f}")
    b.metric("Precision", f"{m['precision']:.2f}")
    c.metric("ROC-AUC", f"{m['roc_auc']:.2f}")
    d.metric("PR-AUC", f"{m['pr_auc']:.2f}")
    st.write(f"Threshold: **{paket['threshold']:.2f}** | Data latih: {paket['n_train']} | Data uji: {paket['n_test']}")
    st.write("**Confusion matrix (data uji)**")
    st.dataframe(pd.DataFrame([[m["TN"], m["FP"]], [m["FN"], m["TP"]]],
                              index=["Aktual Normal", "Aktual BBLR"],
                              columns=["Prediksi Normal", "Prediksi BBLR"]).astype(int))
    st.write("**Perbandingan model (cross-validation, data latih)**")
    st.dataframe(pd.DataFrame(paket["cv_table"]).T.round(3))
    st.write("**Pengaruh fitur** (koefisien LR / importance RF)")
    st.bar_chart(pd.Series(paket["feature_importance"]).head(10))
    with st.expander("Kolom yang dibuang"):
        for k, v in paket["kolom_dibuang"].items():
            st.write(f"- **{k}**: {v}")

Writing app.py


## 7. Jalankan Streamlit di Colab
Memakai Cloudflare Quick Tunnel (tanpa akun/token). Klik link `https://....trycloudflare.com` yang muncul.

In [11]:
import subprocess, time, re, os
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
os.system('pkill -f streamlit; pkill -f cloudflared')
subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port', '8501', '--server.headless', 'true'],
                 stdout=open('streamlit.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(6)
subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:8501'],
                 stdout=open('tunnel.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(30):
    time.sleep(2)
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', open('tunnel.log').read())
    if m: print('Buka aplikasi:', m.group(0)); break
else:
    print(open('tunnel.log').read()[-800:])

Buka aplikasi: https://month-behalf-inherited-princeton.trycloudflare.com


## 8. Laporan singkat (otomatis dari hasil training)

In [12]:
m = paket['metrics_test']; cv = pd.DataFrame(paket['cv_table']).T.round(3)
top = list(paket['feature_importance'].items())[:5]
laporan = f'''# Laporan UTS - Kasus 4: Prediksi Berat Lahir Rendah
NIM 24552011338 | Data: {len(df)} baris mentah, {len(df.drop_duplicates())} setelah duplikat dibuang | Target: low (1 = BBLR, <2500 g)

## 1. Feature Engineering
- Audit: {df.duplicated().sum()} baris duplikat persis dibuang; tidak ada missing; rentang nilai wajar. Kolom `bwt` dibuang (leakage: low = bwt<2500) dan ID dibuang.
- `race` diperlakukan kategorik (kode nominal) lalu one-hot.
- Fitur baru: ptl_pernah, ftv_pernah, usia_berisiko, lwt_rendah, jumlah_risiko (aturan domain tetap, bebas leakage).
- Split 80/20 stratified dilakukan sebelum imputasi/scaling/encoding; semua statistik dari train.

## 2. Perbandingan Model (CV 5x3 di train)
{cv.to_markdown()}
Model terpilih: **{paket['model_name']}** (PR-AUC CV tertinggi di antara model non-baseline).

## 3. Metrik (test, 1 kali, threshold={paket['threshold']:.2f})
Recall={m['recall']:.2f}, Precision={m['precision']:.2f}, F1={m['f1']:.2f}, ROC-AUC={m['roc_auc']:.2f}, PR-AUC={m['pr_auc']:.2f}
Confusion: TN={m['TN']:.0f}, FP={m['FP']:.0f}, FN={m['FN']:.0f}, TP={m['TP']:.0f}
Kesalahan terburuk adalah FN (bayi BBLR terlewat), sehingga metrik utama = recall dengan threshold diturunkan dari 0,5.

## 4. Interpretasi
Fitur paling berpengaruh: {', '.join(f"{k} ({v:+.2f})" for k, v in top)}.
Keterbatasan: data kecil (test hanya {paket['n_test']} baris) sehingga metrik bervariasi; ini alat skrining, bukan diagnosis.
Catatan: selisih PR-AUC kedua model (~0,01) jauh lebih kecil dari simpangan baku CV (~0,09-0,13), jadi keduanya praktis setara; Random Forest dipilih karena skor tertinggi. Specificity test rendah ({m['specificity']:.2f}) adalah harga dari recall tinggi: banyak ibu ditandai berisiko (FP={m['FP']:.0f}) agar hanya {m['FN']:.0f} kasus BBLR terlewat. ROC-AUC {m['roc_auc']:.2f} menunjukkan kemampuan pembeda yang sedang saja.
'''
open('laporan.md', 'w').write(laporan); print(laporan)

# Laporan UTS - Kasus 4: Prediksi Berat Lahir Rendah
NIM 24552011338 | Data: 189 baris mentah, 184 setelah duplikat dibuang | Target: low (1 = BBLR, <2500 g)

## 1. Feature Engineering
- Audit: 5 baris duplikat persis dibuang; tidak ada missing; rentang nilai wajar. Kolom `bwt` dibuang (leakage: low = bwt<2500) dan ID dibuang.
- `race` diperlakukan kategorik (kode nominal) lalu one-hot.
- Fitur baru: ptl_pernah, ftv_pernah, usia_berisiko, lwt_rendah, jumlah_risiko (aturan domain tetap, bebas leakage).
- Split 80/20 stratified dilakukan sebelum imputasi/scaling/encoding; semua statistik dari train.

## 2. Perbandingan Model (CV 5x3 di train)
|                     |   pr_auc |   roc_auc |   recall |   precision |    f1 |   pr_auc_std |
|:--------------------|---------:|----------:|---------:|------------:|------:|-------------:|
| Dummy (baseline)    |    0.32  |     0.5   |    0     |       0     | 0     |        0.011 |
| Logistic Regression |    0.501 |     0.644 |    0.52  |       0.

Simpan notebook ini sebagai `notebook/kasus04_birthwt_colab.ipynb`. Untuk mengumpulkan, unduh `fitur.py`, `latih.py`, `app.py`, `models/model_bblr.joblib`, `laporan.md`.